In [3]:
# ============================================================
# Controlled RF Baseline Experiments
# Reviewer 2 - Comment 3
#
# Experiments:
#   B1: Single-frame, raw EAR only
#   B2: Single-frame, engineered 4-feature representation
#   B3: T=30, raw EAR only
#   B4: T=30, engineered 4-feature representation
#       (validation/reference configuration)
#
# Evaluation:
#   Leave-One-Driver-Out (LODO)
#   Random Forest: 300 trees
#   class_weight="balanced"
#   random_state=42
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

CSV_PATH = "eye_features.csv"

SEED = 42
EAR_THRESHOLD = 0.21
SEQ_LEN = 30

OUTPUT_FOLD = "rf_controlled_baselines_fold_results.csv"
OUTPUT_SUMMARY = "rf_controlled_baselines_summary.csv"


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(CSV_PATH)

required_columns = {"image_name", "ear", "label"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        f"CSV must contain {required_columns}. "
        f"Found: {list(df.columns)}"
    )

print("\nDataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", list(df.columns))


# ------------------------------------------------------------
# 3. PARSE IMAGE FILENAMES
# Exact logic used in the original experimental pipeline
# ------------------------------------------------------------

def parse_filename(name):
    """Return (driver_id, scenario_id, frame_index)."""

    base = os.path.basename(str(name))

    # Driver ID = first three digits before underscore
    m_driver = re.match(r"^(\d{3})_", base)
    driver_id = m_driver.group(1) if m_driver else "unknown"

    stem = os.path.splitext(base)[0]
    parts = stem.split("_")

    # Expected structure:
    # driver + scenario token(s) + frame number + label token
    frame_idx = -1
    frame_pos = None

    # Search backwards for numeric frame-number token
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].isdigit():
            frame_idx = int(parts[i])
            frame_pos = i
            break

    if frame_pos is None:
        scenario = (
            "_".join(parts[1:-1])
            if len(parts) > 2
            else "unknown"
        )
    else:
        scenario = (
            "_".join(parts[1:frame_pos])
            if frame_pos > 1
            else "unknown"
        )

    return driver_id, scenario, frame_idx


parsed = df["image_name"].apply(parse_filename)

df["driver_id"] = parsed.apply(lambda x: x[0])
df["scenario"] = parsed.apply(lambda x: x[1])
df["frame_idx"] = parsed.apply(lambda x: x[2])

df["stream_id"] = (
    df["driver_id"].astype(str)
    + "__"
    + df["scenario"].astype(str)
)

df = df.sort_values(
    ["driver_id", "scenario", "frame_idx", "image_name"]
).reset_index(drop=True)

print("\nDrivers:", sorted(df["driver_id"].unique()))
print("Number of streams:", df["stream_id"].nunique())


# ------------------------------------------------------------
# 4. ENGINEER FEATURES
#
# IMPORTANT:
# Rolling mean and EAR delta are calculated separately
# within each stream so that information cannot cross
# recording boundaries.
# ------------------------------------------------------------

df["ear_roll5"] = (
    df.groupby("stream_id")["ear"]
      .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

df["ear_delta"] = (
    df.groupby("stream_id")["ear"]
      .diff()
      .fillna(0.0)
)

df["eye_closed"] = (df["ear"] < EAR_THRESHOLD).astype(int)

RAW_FEATURES = ["ear"]

ENGINEERED_FEATURES = [
    "ear",
    "ear_roll5",
    "ear_delta",
    "eye_closed"
]

print("\nEngineered features created.")


# ------------------------------------------------------------
# 5. SINGLE-FRAME DATA
# ------------------------------------------------------------

def make_single_frame_data(dataframe, feature_cols):

    X = dataframe[feature_cols].to_numpy(dtype=np.float32)
    y = dataframe["label"].to_numpy(dtype=int)
    drivers = dataframe["driver_id"].to_numpy()

    return X, y, drivers


# ------------------------------------------------------------
# 6. TEMPORAL SEQUENCES
#
# Sequence label = label of final observation in window.
# Windows never cross stream boundaries.
# ------------------------------------------------------------

def make_sequences(dataframe, feature_cols, seq_len=30):

    X_list = []
    y_list = []
    driver_list = []

    for stream_id, gdf in dataframe.groupby(
        "stream_id", sort=False
    ):

        gdf = gdf.sort_values("frame_idx")

        if len(gdf) < seq_len:
            continue

        features = gdf[feature_cols].to_numpy(dtype=np.float32)
        labels = gdf["label"].to_numpy(dtype=int)

        driver = gdf["driver_id"].iloc[0]

        for start in range(0, len(gdf) - seq_len + 1):

            end = start + seq_len

            window = features[start:end]

            # Flatten temporal window for Random Forest
            X_list.append(window.reshape(-1))

            # Target = class of final frame
            y_list.append(labels[end - 1])

            driver_list.append(driver)

    return (
        np.asarray(X_list, dtype=np.float32),
        np.asarray(y_list, dtype=int),
        np.asarray(driver_list)
    )


# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred, y_prob):

    result = {
        "accuracy": accuracy_score(y_true, y_pred),

        "precision_drowsy": precision_score(
            y_true, y_pred,
            pos_label=1,
            zero_division=0
        ),

        "recall_drowsy": recall_score(
            y_true, y_pred,
            pos_label=1,
            zero_division=0
        ),

        "f1_drowsy": f1_score(
            y_true, y_pred,
            pos_label=1,
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        )
    }

    if len(np.unique(y_true)) == 2:
        result["auroc"] = roc_auc_score(y_true, y_prob)
        result["auprc"] = average_precision_score(y_true, y_prob)
    else:
        result["auroc"] = np.nan
        result["auprc"] = np.nan

    return result


# ------------------------------------------------------------
# 8. LODO RANDOM FOREST
#
# IMPORTANT:
# MinMaxScaler is fitted ONLY on the LODO training
# partition and then applied to the held-out driver.
# ------------------------------------------------------------

def run_lodo_rf(X, y, drivers, experiment_name):

    results = []

    unique_drivers = sorted(np.unique(drivers))

    print("\n" + "=" * 70)
    print("EXPERIMENT:", experiment_name)
    print("Samples:", len(y))
    print("Features per sample:", X.shape[1])
    print("=" * 70)

    for test_driver in unique_drivers:

        print(f"\nHeld-out driver: {test_driver}")

        train_mask = drivers != test_driver
        test_mask = drivers == test_driver

        X_train = X[train_mask]
        X_test = X[test_mask]

        y_train = y[train_mask]
        y_test = y[test_mask]

        # Scaling fitted only on training partition
        scaler = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )

        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

        metrics = calculate_metrics(
            y_test,
            y_pred,
            y_prob
        )

        row = {
            "experiment": experiment_name,
            "test_driver": test_driver,
            "n_test": len(y_test),
            **metrics
        }

        results.append(row)

        print(
            f"Accuracy={metrics['accuracy']:.4f} | "
            f"F1={metrics['f1_drowsy']:.4f} | "
            f"AUROC={metrics['auroc']:.4f} | "
            f"AUPRC={metrics['auprc']:.4f}"
        )

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 9. BUILD THE FOUR CONTROLLED CONFIGURATIONS
# ------------------------------------------------------------

print("\nPreparing datasets...")


# B1 - Single frame + raw EAR
X_b1, y_b1, d_b1 = make_single_frame_data(
    df,
    RAW_FEATURES
)


# B2 - Single frame + engineered features
X_b2, y_b2, d_b2 = make_single_frame_data(
    df,
    ENGINEERED_FEATURES
)


# B3 - T=30 + raw EAR
X_b3, y_b3, d_b3 = make_sequences(
    df,
    RAW_FEATURES,
    seq_len=SEQ_LEN
)


# B4 - T=30 + engineered features
X_b4, y_b4, d_b4 = make_sequences(
    df,
    ENGINEERED_FEATURES,
    seq_len=SEQ_LEN
)


print("\nConfiguration sizes:")
print("B1 Single-frame raw EAR:", X_b1.shape)
print("B2 Single-frame engineered:", X_b2.shape)
print("B3 T=30 raw EAR:", X_b3.shape)
print("B4 T=30 engineered:", X_b4.shape)


# ------------------------------------------------------------
# 10. RUN EXPERIMENTS
# ------------------------------------------------------------

all_results = []


# B1
res_b1 = run_lodo_rf(
    X_b1,
    y_b1,
    d_b1,
    "B1_SingleFrame_RawEAR"
)

all_results.append(res_b1)


# B2
res_b2 = run_lodo_rf(
    X_b2,
    y_b2,
    d_b2,
    "B2_SingleFrame_Engineered"
)

all_results.append(res_b2)


# B3
res_b3 = run_lodo_rf(
    X_b3,
    y_b3,
    d_b3,
    "B3_T30_RawEAR"
)

all_results.append(res_b3)


# B4 - validation/reference
res_b4 = run_lodo_rf(
    X_b4,
    y_b4,
    d_b4,
    "B4_T30_Engineered"
)

all_results.append(res_b4)


fold_results = pd.concat(
    all_results,
    ignore_index=True
)


# ------------------------------------------------------------
# 11. CREATE MEAN +/- SD SUMMARY
#
# ddof=1 gives sample SD across the four LODO folds,
# matching the manuscript reporting convention.
# ------------------------------------------------------------

metric_cols = [
    "accuracy",
    "precision_drowsy",
    "recall_drowsy",
    "f1_drowsy",
    "f1_macro",
    "f1_weighted",
    "auroc",
    "auprc"
]

summary_rows = []

for experiment, gdf in fold_results.groupby("experiment"):

    row = {"experiment": experiment}

    for metric in metric_cols:

        row[f"{metric}_mean"] = gdf[metric].mean()

        row[f"{metric}_sd"] = gdf[metric].std(
            ddof=1
        )

    summary_rows.append(row)


summary = pd.DataFrame(summary_rows)


# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

fold_results.to_csv(
    OUTPUT_FOLD,
    index=False
)

summary.to_csv(
    OUTPUT_SUMMARY,
    index=False
)


# ------------------------------------------------------------
# 13. DISPLAY RESULTS
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n\n" + "=" * 70)
print("FOLD-LEVEL RESULTS")
print("=" * 70)

print(
    fold_results.round(4).to_string(index=False)
)


print("\n\n" + "=" * 70)
print("MEAN +/- SD SUMMARY")
print("=" * 70)

for _, row in summary.iterrows():

    print("\n", row["experiment"])

    for metric in metric_cols:

        mean = row[f"{metric}_mean"]
        sd = row[f"{metric}_sd"]

        print(
            f"{metric:20s}: "
            f"{mean:.4f} +/- {sd:.4f}"
        )


print("\n\nResults saved as:")
print("1.", OUTPUT_FOLD)
print("2.", OUTPUT_SUMMARY)

print("\nDONE.")


Dataset loaded successfully.
Rows: 65929
Columns: ['image_name', 'ear', 'label']

Drivers: ['001', '002', '005', '006']
Number of streams: 27

Engineered features created.

Preparing datasets...

Configuration sizes:
B1 Single-frame raw EAR: (65929, 1)
B2 Single-frame engineered: (65929, 4)
B3 T=30 raw EAR: (65146, 30)
B4 T=30 engineered: (65146, 120)

EXPERIMENT: B1_SingleFrame_RawEAR
Samples: 65929
Features per sample: 1

Held-out driver: 001
Accuracy=0.5216 | F1=0.5277 | AUROC=0.5373 | AUPRC=0.5304

Held-out driver: 002
Accuracy=0.4974 | F1=0.5244 | AUROC=0.4941 | AUPRC=0.5728

Held-out driver: 005
Accuracy=0.5133 | F1=0.5513 | AUROC=0.5284 | AUPRC=0.6126

Held-out driver: 006
Accuracy=0.5250 | F1=0.5223 | AUROC=0.5663 | AUPRC=0.4710

EXPERIMENT: B2_SingleFrame_Engineered
Samples: 65929
Features per sample: 4

Held-out driver: 001
Accuracy=0.5365 | F1=0.5698 | AUROC=0.5570 | AUPRC=0.5498

Held-out driver: 002
Accuracy=0.5060 | F1=0.5481 | AUROC=0.5004 | AUPRC=0.5848

Held-out drive